In [2]:
import numpy as np
import matplotlib.pyplot as plt

**4.24 Feedforward 1 layer neural network**

In [11]:
def ReLU(z):
    return np.maximum(0, z)

x = np.array([[1.], [2.]])
W1 = np.array([[1., 0.], [0., 1.]])
W2 = np.array([[1., 1.]])

# hidden layer
z1 = np.dot(W1.T, x)
a1 = ReLU(z1)

# output layer
z2 = np.dot(W2, a1)
a2 = ReLU(z2)

print(f"z1 = {z1.ravel()}, a1 = {a1.ravel()}, z2 = {z2.ravel()}, output = {a2.ravel()}")

z1 = [1. 2.], a1 = [1. 2.], z2 = [3.], output = [3.]


**4.25 Backpropagate with 1 layer neural network**

In [12]:
def drelu(z):
    return (z > 0).astype(float)

y = 5.0
J = 0.5 * (y - a2) ** 2

# backward
dJ_da2 = a2 - y
delta2 = dJ_da2 * drelu(z2)
dW2 = np.dot(delta2, a1.T)
delta1 = np.dot(W2.T, delta2) * drelu(z1)
dW1 = np.dot(delta1, x.T)

print("J =", J.item())
print("dJ/dW1 =\n", dW1)
print("dJ/dW2 =\n", dW2)

J = 2.0
dJ/dW1 =
 [[-2. -4.]
 [-2. -4.]]
dJ/dW2 =
 [[-2. -4.]]


**4.26 Neural Network From Scratch**
- Lựa chọn bài toán phân loại nhị phân
- Số mẫu = 20
- Hàm kích hoạt: ReLU, Sigmoid với 1 output neuron

In [ ]:
# initialize data
m, n_features = 20, 2
X = np.random.randn(n_features, m)
y = (X[0] * X[1] > 0).astype(float).reshape(1, m)

def ReLU(z):
    return np.maximum(0, z)

def drelu(z):
    return (z > 0).astype(float)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def init_params(layer_sizes):
    params = {}
    for l in range(1, len(layer_sizes)):
        params[f"W{l}"] = np.random.randn(layer_sizes[l], layer_sizes[l-1]) * np.sqrt(2 / layer_sizes[l-1])
        params[f"b{l}"] = np.zeros((layer_sizes[l], 1))
    return params

def cross_entropy_loss(A, y):
    eps = 1e-12
    return -np.mean(y * np.log(A + eps) + (1 - y) * np.log(1 - A + eps))

def forward(X, params, L):
    A = X
    cache = {"A0": X}
    for l in range(1, L+1):
        Z = np.dot(params[f"W{l}"], A) + params[f"b{l}"]
        A = sigmoid(Z) if l == L else ReLU(Z)
        cache[f"Z{l}"], cache[f"A{l}"] = Z, A
    return A, cache

def backward(y, params, cache, L):
    m = y.shape[1]
    grads = {}
    dZ = cache[f"A{L}"] - y  # sigmoid + BCE shortcut
    for l in range(L, 0, -1):
        grads[f"dW{l}"] = np.dot(dZ, cache[f"A{l-1}"].T) / m
        grads[f"db{l}"] = np.sum(dZ, axis=1, keepdims=True) / m
        if l > 1:
            dA = np.dot(params[f"W{l}"].T, dZ)
            dZ = dA * drelu(cache[f"Z{l-1}"])
    return grads

def train(X, y, layer_sizes, lr=0.1, epochs=1000):
    L = len(layer_sizes) - 1
    params = init_params(layer_sizes)
    for epoch in range(epochs):
        A, cache = forward(X, params, L)
        loss = cross_entropy_loss(A, y)
        grads = backward(y, params, cache, L)
        for l in range(1, L + 1):
            params[f"W{l}"] -= lr * grads[f"dW{l}"]
            params[f"b{l}"] -= lr * grads[f"db{l}"]
        if epoch % 100 == 0:
            print(f"epoch {epoch:5d}  loss = {loss:.4f}")
    return params

def predict(X, params, L):
    A, _ = forward(X, params, L)
    return (A > 0.5).astype(int)

layer_sizes = [n_features, 8, 4, 1]
params = train(X, y, layer_sizes)

pred = predict(X, params, len(layer_sizes) - 1)
print("Train accuracy:", np.mean(pred == y))

epoch     0  loss = 0.7192
epoch   100  loss = 0.4107
epoch   200  loss = 0.2438
epoch   300  loss = 0.1233
epoch   400  loss = 0.0743
epoch   500  loss = 0.0515
epoch   600  loss = 0.0388
epoch   700  loss = 0.0308
epoch   800  loss = 0.0253
epoch   900  loss = 0.0214
Train accuracy: 1.0
